In [ ]:
# ============================================================
# MACHINE 1 RESUME - DIRECT PATH VERSION
# Resume only rank1 using exact old CSV paths
# ============================================================

# =========================
# 0. CONFIG
# =========================
MACHINE_ID = 0
WORLD_SIZE = 6
RESUME_RANK = 1

# exact old CSV paths provided by user
OLD_RANK0_CSV = "/kaggle/input/<PREVIOUS_RANK_OUTPUTS>/qwen3_full_6gpu/rank0_of_6/qwen3_full_candidates_rank0_of_6.csv"
OLD_RANK1_CSV = "/kaggle/input/<PREVIOUS_RANK_OUTPUTS>/qwen3_full_6gpu/rank1_of_6/qwen3_full_candidates_rank1_of_6.csv"

MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"

TEST_IMAGE_ROOT = "/kaggle/input/<TEST_IMAGE_DATASET>/TEST DATASET/images"
CUI_SUBMISSION_CSV = "/kaggle/input/<CUI_SUBMISSION_DATASET>/submission.csv"
CUI2META_JSON = "/kaggle/input/<CUI_METADATA_DATASET>/cui2meta.json"

BASE_OUT_DIR = "/kaggle/working/qwen3_full_6gpu"

MAX_NEW_TOKENS = 80
MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = 512 * 28 * 28
SAVE_EVERY = 1   # save after every image for safe resume

# use physical GPU 1 for rank1
CUDA_VISIBLE_DEVICE = "1"

# if OOM, reduce this:
# MAX_PIXELS = 448 * 28 * 28

# =========================
# 1. INSTALL
# =========================
import os, sys, subprocess
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = CUDA_VISIBLE_DEVICE
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def sh(cmd):
    print(f"\n[CMD] {cmd}")
    subprocess.check_call(cmd, shell=True)

print("=" * 100)
print("[STEP 1] Install dependencies")
print("=" * 100)

sh(
    f'{sys.executable} -m pip install -q --no-cache-dir --force-reinstall '
    f'"numpy==2.0.2" "pandas==2.2.2" "pillow==11.3.0" "scipy==1.15.3" "scikit-learn==1.6.1"'
)

sh(
    f'{sys.executable} -m pip install -q --no-cache-dir --upgrade '
    f'"accelerate>=1.2.0" "bitsandbytes>=0.43.3" "sentencepiece" "tqdm" '
    f'"qwen-vl-utils==0.0.14" "huggingface-hub>=1.5.0,<2.0" "safetensors>=0.4.5"'
)

sh(
    f'{sys.executable} -m pip install -q --no-cache-dir git+https://github.com/huggingface/transformers'
)

sh(
    f'{sys.executable} -m pip install -q --no-cache-dir "qwen-vl-utils==0.0.14"'
)

# =========================
# 2. IMPORTS
# =========================
print("=" * 100)
print("[STEP 2] Imports")
print("=" * 100)

import re
import gc
import json
import time
import zipfile
import shutil
import warnings

import numpy as np
import pandas as pd
import PIL
import scipy
import sklearn
import torch
import transformers
import huggingface_hub

from tqdm.auto import tqdm
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info

warnings.filterwarnings("ignore")

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("Pillow:", PIL.__version__)
print("scipy:", scipy.__version__)
print("sklearn:", sklearn.__version__)
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Visible GPU count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("Visible GPU name:", torch.cuda.get_device_name(0))

# =========================
# 3. COPY OLD CSV TO WORKING
# =========================
print("=" * 100)
print("[STEP 3] Restore old rank0/rank1 CSV")
print("=" * 100)

OLD_RANK0_CSV = Path(OLD_RANK0_CSV)
OLD_RANK1_CSV = Path(OLD_RANK1_CSV)
TEST_IMAGE_ROOT = Path(TEST_IMAGE_ROOT)
CUI_SUBMISSION_CSV = Path(CUI_SUBMISSION_CSV)
CUI2META_JSON = Path(CUI2META_JSON)

BASE_OUT_DIR = Path(BASE_OUT_DIR)
rank0_dst = BASE_OUT_DIR / "rank0_of_6" / "qwen3_full_candidates_rank0_of_6.csv"
rank1_dst = BASE_OUT_DIR / "rank1_of_6" / "qwen3_full_candidates_rank1_of_6.csv"

rank0_dst.parent.mkdir(parents=True, exist_ok=True)
rank1_dst.parent.mkdir(parents=True, exist_ok=True)

print("OLD_RANK0_CSV:", OLD_RANK0_CSV, OLD_RANK0_CSV.exists())
print("OLD_RANK1_CSV:", OLD_RANK1_CSV, OLD_RANK1_CSV.exists())
print("TEST_IMAGE_ROOT:", TEST_IMAGE_ROOT, TEST_IMAGE_ROOT.exists())
print("CUI_SUBMISSION_CSV:", CUI_SUBMISSION_CSV, CUI_SUBMISSION_CSV.exists())
print("CUI2META_JSON:", CUI2META_JSON, CUI2META_JSON.exists())

if not OLD_RANK0_CSV.exists():
    raise FileNotFoundError(f"Missing OLD_RANK0_CSV: {OLD_RANK0_CSV}")
if not OLD_RANK1_CSV.exists():
    raise FileNotFoundError(f"Missing OLD_RANK1_CSV: {OLD_RANK1_CSV}")

shutil.copy2(OLD_RANK0_CSV, rank0_dst)
shutil.copy2(OLD_RANK1_CSV, rank1_dst)

print("Copied rank0 ->", rank0_dst)
print("Copied rank1 ->", rank1_dst)

# =========================
# 4. QUICK CHECK OLD CSV
# =========================
print("=" * 100)
print("[STEP 4] Check restored CSVs")
print("=" * 100)

rank0_old = pd.read_csv(rank0_dst)
rank1_old = pd.read_csv(rank1_dst)

rank0_old["ID"] = rank0_old["ID"].astype(str)
rank1_old["ID"] = rank1_old["ID"].astype(str)

print("rank0 rows:", len(rank0_old))
print("rank0 empty:", rank0_old["caption_qwen3"].fillna("").astype(str).str.strip().eq("").sum())
print("rank0 duplicate:", rank0_old["ID"].duplicated().sum())
print("rank0 unique captions:", rank0_old["caption_qwen3"].nunique())

print("rank1 rows:", len(rank1_old))
print("rank1 empty:", rank1_old["caption_qwen3"].fillna("").astype(str).str.strip().eq("").sum())
print("rank1 duplicate:", rank1_old["ID"].duplicated().sum())
print("rank1 unique captions:", rank1_old["caption_qwen3"].nunique())

# =========================
# 5. BUILD TARGET SHARD FOR RANK1
# =========================
print("=" * 100)
print("[STEP 5] Build target shard for rank1")
print("=" * 100)

if not TEST_IMAGE_ROOT.exists():
    alt = Path("/kaggle/input/<TEST_IMAGE_DATASET>/TEST DATASET")
    if alt.exists():
        TEST_IMAGE_ROOT = alt

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

def numeric_suffix_value(x):
    s = str(x)
    m = re.search(r"(\d+)(?!.*\d)", s)
    return int(m.group(1)) if m else 10**18

def numeric_suffix_key(series):
    return series.map(numeric_suffix_value)

image_paths = []
for p in TEST_IMAGE_ROOT.rglob("*"):
    if p.suffix.lower() in IMG_EXTS:
        image_paths.append(p)

df_images = pd.DataFrame({
    "ID": [p.stem for p in image_paths],
    "path": [str(p) for p in image_paths],
})

df_images["ID"] = df_images["ID"].astype(str)
df_images = df_images.sort_values("ID", key=numeric_suffix_key).reset_index(drop=True)

# rank 1 in world_size = 6
df_images_rank1 = df_images.iloc[RESUME_RANK::WORLD_SIZE].reset_index(drop=True)

print("Total test images:", len(df_images))
print("Target rank1 images:", len(df_images_rank1))

# load CUI
df_cui = pd.read_csv(CUI_SUBMISSION_CSV)

if "id" in df_cui.columns and "ID" not in df_cui.columns:
    df_cui = df_cui.rename(columns={"id": "ID"})

if "CUIs" not in df_cui.columns:
    possible = [c for c in df_cui.columns if c.lower() in ["cui", "cuis", "concept", "concepts"]]
    if possible:
        df_cui = df_cui.rename(columns={possible[0]: "CUIs"})
    else:
        df_cui["CUIs"] = ""

df_cui["ID"] = df_cui["ID"].astype(str)
df_cui["CUIs"] = df_cui["CUIs"].fillna("").astype(str)

with open(CUI2META_JSON, "r", encoding="utf-8") as f:
    cui2meta = json.load(f)

def cui_to_terms(cui_string, max_terms=8):
    if pd.isna(cui_string):
        return ""
    raw = str(cui_string).strip()
    if not raw:
        return ""
    cuis = re.split(r"[|,; \t\n]+", raw)
    terms = []
    for cui in cuis:
        cui = cui.strip()
        if not cui:
            continue
        meta = cui2meta.get(cui, {})
        if isinstance(meta, dict):
            term = (
                meta.get("term")
                or meta.get("name")
                or meta.get("preferred_name")
                or meta.get("label")
                or ""
            )
        else:
            term = str(meta)
        term = str(term).strip()
        if term and term.lower() not in [t.lower() for t in terms]:
            terms.append(term)
        if len(terms) >= max_terms:
            break
    return "; ".join(terms)

df_cui["cui_terms"] = df_cui["CUIs"].apply(cui_to_terms)

df_rank1 = df_images_rank1.merge(
    df_cui[["ID", "CUIs", "cui_terms"]],
    on="ID",
    how="left"
)
df_rank1["CUIs"] = df_rank1["CUIs"].fillna("")
df_rank1["cui_terms"] = df_rank1["cui_terms"].fillna("")

# =========================
# 6. FIND MISSING IDS
# =========================
print("=" * 100)
print("[STEP 6] Find missing rows")
print("=" * 100)

rank1_old = rank1_old.drop_duplicates("ID", keep="first").reset_index(drop=True)
rank1_old["caption_qwen3"] = rank1_old["caption_qwen3"].fillna("").astype(str)

done_ids = set(rank1_old.loc[rank1_old["caption_qwen3"].str.strip() != "", "ID"])
missing_df = df_rank1[~df_rank1["ID"].astype(str).isin(done_ids)].copy()

print("Existing rank1 rows:", len(rank1_old))
print("Target rank1 rows:", len(df_rank1))
print("Done valid rows:", len(done_ids))
print("Missing rows to generate:", len(missing_df))
print("Preview missing IDs:")
print(missing_df[["ID"]].head(30))

# =========================
# 7. CLEANER + PROMPT
# =========================
BAD_PREFIXES = [
    r"^caption\s*:\s*",
    r"^answer\s*:\s*",
    r"^output\s*:\s*",
    r"^final caption\s*:\s*",
    r"^this image shows\s+",
    r"^the image shows\s+",
    r"^this figure shows\s+",
    r"^the figure shows\s+",
    r"^an image of\s+",
    r"^a medical image of\s+",
]

GENERIC_BAD = {
    "the relevant anatomical region",
    "relevant anatomical region",
    "medical image showing findings",
    "the image shows the relevant anatomy",
    "an image of the relevant anatomical region",
    "medical imaging figure",
    "medical image",
}

MODALITY_REPLACEMENTS = [
    (r"\bX-Ray Computed Tomography\b", "CT"),
    (r"\bComputed Tomography\b", "CT"),
    (r"\bcomputed tomography\b", "CT"),
    (r"\bComputerized Tomography\b", "CT"),
    (r"\bMagnetic Resonance Imaging\b", "MRI"),
    (r"\bmagnetic resonance imaging\b", "MRI"),
    (r"\bUltrasonography\b", "ultrasound"),
    (r"\bultrasonography\b", "ultrasound"),
    (r"\bSonography\b", "ultrasound"),
    (r"\bsonography\b", "ultrasound"),
    (r"\bPlain x-ray\b", "X-ray"),
    (r"\bplain x-ray\b", "X-ray"),
    (r"\bplain radiograph\b", "X-ray"),
]

TRUNCATED_ENDINGS = [
    r"\blikely\.$",
    r"\bsuggestive of\.$",
    r"\bindicative of\.$",
    r"\bconsistent with\.$",
    r"\bwith no\.$",
    r"\blocated in the\.$",
    r"\bat the\.$",
    r"\bdemonstrating\.$",
    r"\bshowing\.$",
]

def clean_caption_v4(text):
    if pd.isna(text):
        return ""
    s = str(text).strip()
    s = s.replace("\n", " ")
    s = re.sub(r"\s+", " ", s).strip()
    s = s.strip("`").strip()
    s = s.strip("\"'“”‘’").strip()

    for pat in BAD_PREFIXES:
        s = re.sub(pat, "", s, flags=re.IGNORECASE).strip()

    s = re.sub(r"(?i)^write one concise.*?caption[:\s-]*", "", s).strip()
    s = re.sub(r"(?i)^generate one concise.*?caption[:\s-]*", "", s).strip()
    s = re.sub(r"(?i)^final caption[:\s-]*", "", s).strip()

    for pat, rep in MODALITY_REPLACEMENTS:
        s = re.sub(pat, rep, s, flags=re.IGNORECASE)

    s = re.sub(r"\s+([,.;:!?])", r"\1", s)
    s = re.sub(r"\s+", " ", s).strip()

    words = s.split()
    if len(words) > 55:
        sentences = re.split(r"(?<=[.!?])\s+", s)
        if len(sentences) > 1:
            s = sentences[0].strip()
            if len(s.split()) < 8 and len(sentences) > 1:
                s = (s + " " + sentences[1]).strip()
        else:
            s = " ".join(words[:55]).strip()

    for pat in TRUNCATED_ENDINGS:
        if re.search(pat, s, flags=re.IGNORECASE):
            s = re.sub(pat, ".", s, flags=re.IGNORECASE).strip()

    s = s.strip()

    if s.lower().strip(". ") in GENERIC_BAD:
        return ""

    if s and s[-1] not in ".!?":
        s += "."

    return s

def build_qwen3_prompt(cui_terms):
    hint = ""
    if cui_terms and len(str(cui_terms).strip()) > 0:
        hint = f"""
Possible UMLS/CUI clinical hints: {cui_terms}
Use these hints only if they are visually supported by the image.
Do not force unsupported diseases or findings.
"""
    return f"""
You are generating a caption for a medical figure in a biomedical article.

Task:
Write one concise, clinically relevant English caption for the image.

Rules:
- Focus on the visible medical content.
- Mention modality when clear: CT, MRI, X-ray, ultrasound, histology, endoscopy, angiography, PET/CT, microscopy.
- Mention anatomical region and main visual finding if visible.
- Do not write "This image shows".
- Do not write "Caption:".
- Do not include uncertainty unless needed.
- Do not invent patient age, sex, diagnosis, treatment, or outcome if not visible.
- Output only one caption sentence.

{hint}
Final caption:
""".strip()

# =========================
# 8. RESUME ONLY MISSING
# =========================
print("=" * 100)
print("[STEP 8] Resume rank1")
print("=" * 100)

records = rank1_old.to_dict("records")

if len(missing_df) == 0:
    print("Rank1 already complete. No need to load model.")
else:
    print("Loading model...")

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )

    processor = AutoProcessor.from_pretrained(
        MODEL_ID,
        min_pixels=MIN_PIXELS,
        max_pixels=MAX_PIXELS,
        trust_remote_code=True,
    )

    model.eval()

    try:
        print("model.device:", model.device)
    except Exception:
        print("first param device:", next(model.parameters()).device)

    @torch.inference_mode()
    def generate_caption(image_path, cui_terms="", max_new_tokens=80):
        prompt = build_qwen3_prompt(cui_terms)

        messages = [
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "image": str(image_path),
                        "min_pixels": MIN_PIXELS,
                        "max_pixels": MAX_PIXELS,
                    },
                    {"type": "text", "text": prompt},
                ],
            }
        ]

        text = processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        image_inputs, video_inputs = process_vision_info(messages)

        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        )

        try:
            inputs = inputs.to(model.device)
        except Exception:
            first_device = next(model.parameters()).device
            inputs = {k: v.to(first_device) if hasattr(v, "to") else v for k, v in inputs.items()}

        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
        )

        input_len = inputs["input_ids"].shape[-1]
        generated_ids_trimmed = generated_ids[:, input_len:]

        output_text = processor.batch_decode(
            generated_ids_trimmed,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )[0]

        return output_text.strip()

    start = time.time()

    for _, row in tqdm(missing_df.iterrows(), total=len(missing_df), desc="resume_rank1"):
        img_id = str(row["ID"])

        try:
            t0 = time.time()
            raw = generate_caption(row["path"], row["cui_terms"], MAX_NEW_TOKENS)
            sec = time.time() - t0
            clean = clean_caption_v4(raw)
        except Exception as e:
            print("[ERROR]", img_id, repr(e))
            raw = ""
            clean = ""
            sec = -1

        records.append({
            "ID": img_id,
            "path": row["path"],
            "CUIs": row["CUIs"],
            "cui_terms": row["cui_terms"],
            "caption_qwen3_raw": raw,
            "caption_qwen3": clean,
            "seconds": sec,
        })

        # save after every image
        tmp = pd.DataFrame(records)
        tmp["ID"] = tmp["ID"].astype(str)
        tmp = tmp.drop_duplicates("ID", keep="first")
        tmp = tmp.sort_values("ID", key=numeric_suffix_key).reset_index(drop=True)
        tmp.to_csv(rank1_dst, index=False)

        print("Saved rows:", len(tmp), "/", len(df_rank1))

    elapsed = (time.time() - start) / 60
    print("Resume elapsed minutes:", round(elapsed, 2))

# =========================
# 9. FINAL CHECK
# =========================
print("=" * 100)
print("[STEP 9] Final check")
print("=" * 100)

final_rank0 = pd.read_csv(rank0_dst)
final_rank1 = pd.read_csv(rank1_dst)

final_rank0["ID"] = final_rank0["ID"].astype(str)
final_rank1["ID"] = final_rank1["ID"].astype(str)

final_rank0 = final_rank0.drop_duplicates("ID", keep="first")
final_rank0 = final_rank0.sort_values("ID", key=numeric_suffix_key).reset_index(drop=True)
final_rank0.to_csv(rank0_dst, index=False)

final_rank1 = final_rank1.drop_duplicates("ID", keep="first")
final_rank1 = final_rank1.sort_values("ID", key=numeric_suffix_key).reset_index(drop=True)
final_rank1.to_csv(rank1_dst, index=False)

print("rank0 rows:", len(final_rank0))
print("rank0 empty:", final_rank0["caption_qwen3"].fillna("").astype(str).str.strip().eq("").sum())
print("rank0 duplicate:", final_rank0["ID"].duplicated().sum())

print("rank1 rows:", len(final_rank1))
print("rank1 target:", len(df_rank1))
print("rank1 empty:", final_rank1["caption_qwen3"].fillna("").astype(str).str.strip().eq("").sum())
print("rank1 duplicate:", final_rank1["ID"].duplicated().sum())
print("rank1 unique captions:", final_rank1["caption_qwen3"].nunique())

machine1 = pd.concat([final_rank0, final_rank1], ignore_index=True)
print("\nMachine 1 combined rows:", len(machine1))
print("Machine 1 empty:", machine1["caption_qwen3"].fillna("").astype(str).str.strip().eq("").sum())
print("Machine 1 duplicate IDs:", machine1["ID"].astype(str).duplicated().sum())

# =========================
# 10. ZIP OUTPUT
# =========================
print("=" * 100)
print("[STEP 10] Zip Machine 1 result")
print("=" * 100)

zip_out = Path("/kaggle/working/machine1_resume_result.zip")

with zipfile.ZipFile(zip_out, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(rank0_dst, arcname="qwen3_full_6gpu/rank0_of_6/qwen3_full_candidates_rank0_of_6.csv")
    z.write(rank1_dst, arcname="qwen3_full_6gpu/rank1_of_6/qwen3_full_candidates_rank1_of_6.csv")

print("Saved zip:", zip_out)
print("=" * 100)
print("DONE MACHINE 1 RESUME")
print("=" * 100)